In [ ]:
# Import modules
import paramiko
import os
from stat import S_ISDIR, S_ISREG
import pandas as pd
from abc import ABC, abstractmethod


In [ ]:
class AVL_FIRE_RemoteDataReader(ABC):
    def __init__(self, sftp_client, project_directory, model_name, local_directory=None):
        self.sftp = sftp_client
        self.project_directory = project_directory
        self.model_name = model_name
        self.local_directory = local_directory
        self.data = None

    @abstractmethod
    def read_data_file(self, file_handle):
        pass

    
    def get_data(self):
        pass

    def get_data_for_all_cases(self, data_sub_dir, file_extension):
        data_dict = {}
        simulation_directory = self.project_directory + "/simulation/project"
        for entry in self.sftp.listdir_attr(simulation_directory):
            case_path = simulation_directory + "/" + entry.filename
            mode = entry.st_mode
            if S_ISDIR(mode):
                sub_paths = self.sftp.listdir(case_path)
            if 'results' in sub_paths:
                data_path = case_path + "/" + data_sub_dir + "/" + self.model_name + '.' + file_extension

                # Test if file is found
                try:
                    file_stat = self.sftp.stat(data_path)
                    print(f"File size: {file_stat.st_size} bytes")
                except FileNotFoundError:
                    print(f"File {data_path} not found on remote server")
                    continue                
                # Get file or data
                if self.local_directory is not None:
                    self.sftp.get(data_path, os.path.join(self.local_directory, 
                                                          entry.filename + "_" + self.model_name + '.csv'))
                else:
                    with self.sftp.open(data_path, 'r') as data_file:
                        # data = remote_file.read()
                        data = self.read_data_file(data_file)
                    data_dict[entry.filename] = data 
        return data_dict
    
    
class AVL_FIRE_2DResultsReader(AVL_FIRE_RemoteDataReader):

    def read_data_file(self, file_handle):
        # Read CSV data into a pandas DataFrame
        data = pd.read_csv(file_handle, skiprows=0, header=[1, 2], sep=";")
        return data
    
    def get_data(self):
        self.data = self.get_data_for_all_cases(data_sub_dir="results", file_extension="csv")
        return self.data


class AVL_FIRE_ParametersReader(AVL_FIRE_RemoteDataReader):

    def read_data_file(self, file_handle):
        # Read CSV data into a pandas DataFrame
        data = pd.read_csv(file_handle, skiprows=0, header=[1, 2], sep=";")
        return data        

In [ ]:
# Function to search for file paths recursively in subfolders
def get_ssh_avl_fire_2d_results(sftp, project_directory, model_name, local_directory=None):
    data_dict = {}
    simulation_directory = project_directory + "/simulation/project"
    for entry in sftp.listdir_attr(simulation_directory):
        remote_path = simulation_directory + "/" + entry.filename
        mode = entry.st_mode
        if S_ISDIR(mode):
            sub_paths = sftp.listdir(remote_path)
            if 'results' in sub_paths:
                remote_results_path = remote_path + "/results/" + model_name + '.csv'

                # Test if file is found
                try:
                    file_stat = sftp.stat(remote_results_path)
                    print(f"File size: {file_stat.st_size} bytes")
                except FileNotFoundError:
                    print(f"File {remote_results_path} not found on remote server")
                    continue                
                # Get file or data
                if local_directory is not None:
                    sftp.get(remote_results_path, os.path.join(local_directory, entry.filename + "_" + model_name + '.csv'))
                else:
                    with sftp.open(remote_results_path, 'r') as remote_file:
                        # data = remote_file.read()
                        data = pd.read_csv(remote_file, skiprows=0, header=[1, 2], sep=";")
                    data_dict[entry.filename] = data 
    return data_dict


In [ ]:
def get_ssh_avl_fire_model_parameters(sftp, project_directory, model_name):
    param_dict = {}
    simulation_directory = project_directory + "/simulation/project"
    for entry in sftp.listdir_attr(simulation_directory):
        remote_path = simulation_directory + "/" + entry.filename
        mode = entry.st_mode
        if S_ISDIR(mode):
            sub_paths = sftp.listdir(remote_path)
            if 'results' in sub_paths:
                remote_results_path = remote_path + "/results/" + model_name + '_model_parameters.csv'

                # Test if file is found
                try:
                    file_stat = sftp.stat(remote_results_path)
                    print(f"File size: {file_stat.st_size} bytes")
                except FileNotFoundError:
                    print(f"File {remote_results_path} not found on remote server")
                    continue                
                # Get file or data
                with sftp.open(remote_results_path, 'r') as remote_file:
                    data = pd.read_csv(remote_file, skiprows=0, header=0, sep=";")
                param_dict[entry.filename] = data 
    return param_dict

In [ ]:
# Copy all files to local folder (slower)
# data_frames = get_ssh_avl_fire_2d_results(sftp_client, PROJECT_DIRECTORY, MODEL_NAME, local_directory="T:\Feierabend\Test")
# Get data into pandas dataframes
data_2d = get_ssh_avl_fire_2d_results(sftp_client, PROJECT_DIRECTORY, MODEL_NAME)



In [ ]:
# Get data into pandas dataframes with OOP implementation
results_reader = AVL_FIRE_2DResultsReader(sftp_client, PROJECT_DIRECTORY, MODEL_NAME)
data_2d_oop = results_reader.get_data()

In [ ]:
# Close the connections
if 'sftp_client' in locals() and sftp_client:
    sftp_client.close()
    print("SFTP session closed.")
if 'ssh_client' in locals() and ssh_client:
    ssh_client.close()
    print("SSH connection closed.")

In [ ]:
data_2d_oop